In [2]:
import pandas as pd
import numpy as np
import os
import json
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
print("successfully installed")

successfully installed


In [3]:
# Corrected path - your file is in data/ not data/processed/
df = pd.read_csv('../data/filtered_complaints.csv')

In [4]:
df.shape

(80667, 23)

In [5]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count',
       'char_count', 'cleaned_narrative', 'cleaned_word_count',
       'cleaned_char_count'],
      dtype='object')

In [8]:
product_counts = df['Product'].value_counts()
for product, count in product_counts.items():
    print(f"  {product}: {count:,} ({count/len(df)*100:.1f}%)")

  Credit card: 80,667 (100.0%)


In [10]:
has_narrative = df['cleaned_narrative'].notna() & (df['cleaned_narrative'].str.len() > 0)
print(f"\n Narratives available: {has_narrative.sum():,}/{len(df):,} ({has_narrative.sum()/len(df)*100:.1f}%)")

# Display first few rows
print("\n First 3 rows:")
display(df[['Product', 'cleaned_narrative']].head(3))


 Narratives available: 80,667/80,667 (100.0%)

 First 3 rows:


,Product,cleaned_narrative
0,Credit card,a xxxx xxxx card was opened under my name by a...
1,Credit card,"dear cfpb, i have a secured credit card with c..."
2,Credit card,i have a citi rewards cards. the credit balanc...


In [27]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

print("LOCAL EMBEDDING MODEL - NO DOWNLOADS")

class LocalEmbeddingModel:
    """
    Local embedding model using TF-IDF with SVD dimension reduction.
    No internet downloads required.
    """
    def __init__(self, dimension=384, max_features=5000):
        self.dimension = dimension
        self.max_features = max_features
        self.vectorizer = None
        self.svd = None
        self.is_fitted = False
        
    def fit(self, texts):
        """Fit the model on text data"""
        print("\n Fitting embedding model on text data...")
        
        # Step 1: Create TF-IDF vectors
        print("  - Creating TF-IDF vectors...")
        self.vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            stop_words='english',
            lowercase=True,
            ngram_range=(1, 2)
        )
        tfidf_matrix = self.vectorizer.fit_transform(texts)
        print(f"    TF-IDF shape: {tfidf_matrix.shape}")
        
        # Step 2: Reduce dimension using SVD
        print("  - Reducing dimension with SVD...")
        n_components = min(self.dimension, tfidf_matrix.shape[1] - 1)
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        reduced_matrix = self.svd.fit_transform(tfidf_matrix)
        print(f"    Reduced shape: {reduced_matrix.shape}")
        
        # Step 3: Pad if needed
        if reduced_matrix.shape[1] < self.dimension:
            padded = np.zeros((reduced_matrix.shape[0], self.dimension))
            padded[:, :reduced_matrix.shape[1]] = reduced_matrix
            self.embedding_matrix = padded
        else:
            self.embedding_matrix = reduced_matrix
        
        # Step 4: Normalize
        self.embedding_matrix = normalize(self.embedding_matrix, norm='l2')
        
        self.is_fitted = True
        print(f"  Embedding matrix shape: {self.embedding_matrix.shape}")
        print(" Model fitted successfully!")
        
    def encode(self, texts, normalize_embeddings=True):
        """Generate embeddings for texts"""
        if isinstance(texts, str):
            texts = [texts]
        
        if not self.is_fitted:
            # Fit on the fly
            self.fit(texts)
        
        # Transform using TF-IDF
        tfidf_vectors = self.vectorizer.transform(texts)
        
        # Apply SVD
        reduced = self.svd.transform(tfidf_vectors)
        
        # Pad if needed
        if reduced.shape[1] < self.dimension:
            padded = np.zeros((reduced.shape[0], self.dimension))
            padded[:, :reduced.shape[1]] = reduced
            embeddings = padded
        else:
            embeddings = reduced
        
        # Normalize
        if normalize_embeddings:
            embeddings = normalize(embeddings, norm='l2')
        
        return embeddings.astype(np.float32)
    
    def get_sentence_embedding_dimension(self):
        return self.dimension

# Create and use the model
embedding_model = LocalEmbeddingModel(dimension=384)

# Test with sample texts

print("TESTING LOCAL EMBEDDING MODEL")

sample_texts = [
    "This is a test complaint about credit card billing",
    "I have an issue with my personal loan payment",
    "Money transfer failed and funds were lost"
]

print("\n Generating embeddings for sample texts...")
embeddings = embedding_model.encode(sample_texts)

print(f" Test embedding shape: {embeddings.shape}")
print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")
print(f"   Sample embedding (first 10): {embeddings[0][:10]}")

print("\n Model ready for Task 2!")

LOCAL EMBEDDING MODEL - NO DOWNLOADS
TESTING LOCAL EMBEDDING MODEL

 Generating embeddings for sample texts...

 Fitting embedding model on text data...
  - Creating TF-IDF vectors...
    TF-IDF shape: (3, 25)
  - Reducing dimension with SVD...
    Reduced shape: (3, 3)
  Embedding matrix shape: (3, 384)
 Model fitted successfully!
 Test embedding shape: (3, 384)
   Embedding dimension: 384
   Sample embedding (first 10): [9.2391277e-17 1.0000000e+00 2.4566786e-17 0.0000000e+00 0.0000000e+00
 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00]

 Model ready for Task 2!


In [28]:

import pandas as pd
import numpy as np
import os
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

print("TASK 2: LOCAL EMBEDDINGS + FAISS VECTOR STORE")


TASK 2: LOCAL EMBEDDINGS + FAISS VECTOR STORE


In [29]:

print("LOADING CLEANED DATASET")

# Try multiple paths
filepath = '../data/filtered_complaints.csv'
if not os.path.exists(filepath):
    filepath = 'data/filtered_complaints.csv'
if not os.path.exists(filepath):
    filepath = 'filtered_complaints.csv'

df = pd.read_csv(filepath)
print(f" Dataset loaded from: {filepath}")
print(f" Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\n Columns: {df.columns.tolist()}")

# Check product distribution
print(f"\n Product Distribution:")
product_counts = df['Product'].value_counts()
for product, count in product_counts.items():
    print(f"  {product}: {count:,} ({count/len(df)*100:.1f}%)")

# Check cleaned narrative quality
has_narrative = df['cleaned_narrative'].notna() & (df['cleaned_narrative'].str.len() > 0)
print(f"\n Narratives available: {has_narrative.sum():,}/{len(df):,} ({has_narrative.sum()/len(df)*100:.1f}%)")

LOADING CLEANED DATASET
 Dataset loaded from: ../data/filtered_complaints.csv
 Shape: 80,667 rows, 23 columns

 Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count', 'char_count', 'cleaned_narrative', 'cleaned_word_count', 'cleaned_char_count']

 Product Distribution:
  Credit card: 80,667 (100.0%)

 Narratives available: 80,667/80,667 (100.0%)


In [30]:

print("STRATIFIED SAMPLING")


def create_stratified_sample(df, sample_size=12000, product_col='Product', random_state=42):
    """
    Create a stratified sample with proportional representation
    """
    product_counts = df[product_col].value_counts()
    sample_fraction = sample_size / len(df)
    
    # Calculate samples per product
    samples_per_product = {}
    print(f"\n Sample allocation:")
    for product, count in product_counts.items():
        n_samples = max(1, int(count * sample_fraction))
        samples_per_product[product] = n_samples
        print(f"  {product}: {count:,} → {n_samples:,} samples ({n_samples/count*100:.1f}%)")
    
    # Adjust total if needed
    total_samples = sum(samples_per_product.values())
    if total_samples > sample_size:
        while total_samples > sample_size:
            max_product = max(samples_per_product, key=lambda x: samples_per_product[x])
            if samples_per_product[max_product] > 1:
                samples_per_product[max_product] -= 1
                total_samples -= 1
            else:
                break
    
    # Sample from each product
    sampled_dfs = []
    print(f"\n Sampling from each product:")
    for product, n in samples_per_product.items():
        product_df = df[df[product_col] == product]
        n_samples = min(n, len(product_df))
        if n_samples > 0:
            sampled = product_df.sample(n=n_samples, random_state=random_state)
            sampled_dfs.append(sampled)
            print(f"  {product}: {n_samples:,} samples taken")
    
    sampled_df = pd.concat(sampled_dfs).reset_index(drop=True)
    
    print(f"\n Sampling complete!")
    print(f"  Target sample size: {sample_size:,}")
    print(f"  Actual sample size: {len(sampled_df):,}")
    print(f"\n Sample product distribution:")
    for product, count in sampled_df[product_col].value_counts().items():
        print(f"  {product}: {count:,} ({count/len(sampled_df)*100:.1f}%)")
    
    return sampled_df

# Create the stratified sample
SAMPLE_SIZE = 12000
sampled_df = create_stratified_sample(df, sample_size=SAMPLE_SIZE)

# Save the sample
os.makedirs('../data/processed', exist_ok=True)
sampled_df.to_csv('../data/processed/stratified_sample.csv', index=False)
print(f"\n Sample saved to: ../data/processed/stratified_sample.csv")

STRATIFIED SAMPLING

 Sample allocation:
  Credit card: 80,667 → 12,000 samples (14.9%)

 Sampling from each product:
  Credit card: 12,000 samples taken

 Sampling complete!
  Target sample size: 12,000
  Actual sample size: 12,000

 Sample product distribution:
  Credit card: 12,000 (100.0%)

 Sample saved to: ../data/processed/stratified_sample.csv


In [31]:

print("TEXT CHUNKING")
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_texts(texts, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """
    Chunk texts using RecursiveCharacterTextSplitter
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
    )
    
    all_chunks = []
    for idx, text in enumerate(tqdm(texts, desc="Chunking texts")):
        if pd.isna(text) or text == '':
            continue
        chunks = text_splitter.split_text(text)
        for chunk_idx, chunk in enumerate(chunks):
            all_chunks.append({
                'chunk': chunk,
                'chunk_index': chunk_idx,
                'total_chunks': len(chunks),
                'original_index': idx,
            })
    return all_chunks

# Extract texts and apply chunking
texts = sampled_df['cleaned_narrative'].tolist()
print(f" Processing {len(texts):,} documents...")

all_chunks = chunk_texts(texts)

# Create chunks dataframe with metadata
chunks_df = pd.DataFrame(all_chunks)
original_indices = chunks_df['original_index'].values
chunks_df['complaint_id'] = sampled_df.iloc[original_indices]['Complaint ID'].values
chunks_df['product'] = sampled_df.iloc[original_indices]['Product'].values
chunks_df['issue'] = sampled_df.iloc[original_indices]['Issue'].values
chunks_df['company'] = sampled_df.iloc[original_indices]['Company'].values
chunks_df['date_received'] = sampled_df.iloc[original_indices]['Date received'].values

# Drop original_index
chunks_df = chunks_df.drop('original_index', axis=1)

print(f"\nChunking complete!")
print(f"  Total documents: {len(texts):,}")
print(f"  Total chunks: {len(chunks_df):,}")
print(f"  Avg chunks per document: {len(chunks_df)/len(texts):.1f}")

# Display sample
print(f"\nSam ple chunks:")
display(chunks_df[['product', 'chunk', 'chunk_index', 'total_chunks']].head(3))

# Save chunks
chunks_df.to_csv('../data/processed/chunks.csv', index=False)
print(f"\n Chunks saved to: ../data/processed/chunks.csv")

TEXT CHUNKING
 Processing 12,000 documents...


Chunking texts: 100%|██████████| 12000/12000 [00:00<00:00, 30318.53it/s]



Chunking complete!
  Total documents: 12,000
  Total chunks: 36,399
  Avg chunks per document: 3.0

Sam ple chunks:


,product,chunk,chunk_index,total_chunks
0,Credit card,i have a balance of 710.00 from my xxxx xxxx x...,0,2
1,Credit card,". wayfair directed me to xxxx and, xxxx direct...",1,2
2,Credit card,my original credit card rate on my american ex...,0,2



 Chunks saved to: ../data/processed/chunks.csv


In [32]:

print("GENERATING EMBEDDINGS")

# Fit the local embedding model on all chunks
print(" Fitting embedding model on all chunks...")
chunk_texts = chunks_df['chunk'].tolist()

# Fit the model (if not already fitted)
embedding_model.fit(chunk_texts)
print(f" Model fitted on {len(chunk_texts):,} chunks")

# Generate embeddings in batches to manage memory
print("\n Generating embeddings in batches...")
batch_size = 64
all_embeddings = []

for i in tqdm(range(0, len(chunk_texts), batch_size), desc="Generating embeddings"):
    batch = chunk_texts[i:i+batch_size]
    batch_embeddings = embedding_model.encode(batch)
    all_embeddings.append(batch_embeddings)

# Combine all embeddings
embeddings = np.vstack(all_embeddings)

print(f"\n Embeddings generated!")
print(f"  Shape: {embeddings.shape}")
print(f"  Memory usage: {embeddings.nbytes / (1024*1024):.2f} MB")

# Save embeddings
np.save('../data/processed/embeddings.npy', embeddings)
print(f" Embeddings saved to: ../data/processed/embeddings.npy")

GENERATING EMBEDDINGS
 Fitting embedding model on all chunks...

 Fitting embedding model on text data...
  - Creating TF-IDF vectors...
    TF-IDF shape: (36399, 5000)
  - Reducing dimension with SVD...
    Reduced shape: (36399, 384)
  Embedding matrix shape: (36399, 384)
 Model fitted successfully!
 Model fitted on 36,399 chunks

 Generating embeddings in batches...


Generating embeddings: 100%|██████████| 569/569 [00:10<00:00, 55.99it/s]



 Embeddings generated!
  Shape: (36399, 384)
  Memory usage: 53.32 MB
 Embeddings saved to: ../data/processed/embeddings.npy


In [33]:

print("CREATING FAISS VECTOR STORE")

def create_faiss_index(embeddings, metadata_df):
    """
    Create a FAISS index with metadata
    """
    # Get embedding dimension
    dimension = embeddings.shape[1]
    print(f"\n Embedding dimension: {dimension}")
    
    # Create FAISS index (using L2 distance)
    print("Creating FAISS index with L2 distance...")
    index = faiss.IndexFlatL2(dimension)
    
    # Add embeddings to index
    print(" Adding embeddings to index...")
    index.add(embeddings.astype('float32'))
    
    print(f" FAISS index created with {index.ntotal:,} vectors")
    
    # Create metadata dictionary for retrieval
    print(" Creating metadata mapping...")
    metadata_dict = {
        'chunk_text': metadata_df['chunk'].tolist(),
        'complaint_id': metadata_df['complaint_id'].tolist(),
        'product': metadata_df['product'].tolist(),
        'issue': metadata_df['issue'].tolist(),
        'company': metadata_df['company'].tolist(),
        'chunk_index': metadata_df['chunk_index'].tolist(),
        'total_chunks': metadata_df['total_chunks'].tolist(),
        'date_received': metadata_df['date_received'].tolist(),
    }
    
    return index, metadata_dict

# Create FAISS index
faiss_index, faiss_metadata = create_faiss_index(embeddings, chunks_df)

# Save FAISS index and metadata
os.makedirs('../vector_store/faiss/', exist_ok=True)

# Save FAISS index
faiss.write_index(faiss_index, '../vector_store/faiss/index.faiss')
print(f"\n FAISS index saved to: ../vector_store/faiss/index.faiss")

# Save metadata
with open('../vector_store/faiss/metadata.pkl', 'wb') as f:
    pickle.dump(faiss_metadata, f)
print(f" Metadata saved to: ../vector_store/faiss/metadata.pkl")

# Save model info
with open('../vector_store/faiss/model_info.txt', 'w') as f:
    f.write("="*60 + "\n")
    f.write("FAISS VECTOR STORE INFORMATION\n")
    f.write("="*60 + "\n\n")
    f.write(f"Model: Local TF-IDF + SVD\n")
    f.write(f"Dimension: {embeddings.shape[1]}\n")
    f.write(f"Total vectors: {embeddings.shape[0]:,}\n")
    f.write(f"Chunk size: {CHUNK_SIZE}\n")
    f.write(f"Chunk overlap: {CHUNK_OVERLAP}\n")
    f.write(f"Total documents: {len(sampled_df):,}\n")
    f.write(f"Average chunks per doc: {len(chunks_df)/len(sampled_df):.1f}\n")
    f.write(f"Index type: IndexFlatL2 (L2 distance)\n")
    f.write(f"Date created: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*60 + "\n")

print(f" Model info saved to: ../vector_store/faiss/model_info.txt")
print(f"\n FAISS vector store complete!")

CREATING FAISS VECTOR STORE

 Embedding dimension: 384
Creating FAISS index with L2 distance...
 Adding embeddings to index...
 FAISS index created with 36,399 vectors
 Creating metadata mapping...

 FAISS index saved to: ../vector_store/faiss/index.faiss
 Metadata saved to: ../vector_store/faiss/metadata.pkl
 Model info saved to: ../vector_store/faiss/model_info.txt

 FAISS vector store complete!


In [34]:

print("TESTING VECTOR STORE RETRIEVAL")
def test_retrieval(query, index, metadata, model, k=5):
    """
    Test retrieval from the vector store
    """
    # Generate query embedding
    query_embedding = model.encode([query])
    
    # Search the index
    distances, indices = index.search(query_embedding.astype('float32'), k)
    
    # Get results
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(metadata['chunk_text']):
            results.append({
                'chunk': metadata['chunk_text'][idx],
                'complaint_id': metadata['complaint_id'][idx],
                'product': metadata['product'][idx],
                'issue': metadata['issue'][idx],
                'company': metadata['company'][idx],
                'distance': distances[0][i],
                'similarity': 1 / (1 + distances[0][i])
            })
    
    return results

# Test queries
test_queries = [
    "What are the most common complaints about credit cards?",
    "Why are customers unhappy with money transfers?",
    "Tell me about billing disputes",
    "What issues do customers face with personal loans?",
]

print("\n Testing retrieval with sample queries:")

for query in test_queries:
    print(f"\n Query: {query}")
    results = test_retrieval(query, faiss_index, faiss_metadata, embedding_model, k=3)
    
    print(f"  Top {len(results)} results:")
    for i, result in enumerate(results, 1):
        print(f"    {i}. Product: {result['product']}")
        print(f"       Issue: {result['issue']}")
        print(f"       Similarity: {result['similarity']:.4f}")
        print(f"       Preview: {result['chunk'][:100]}...")
        print()

print("\n Retrieval testing complete!")

TESTING VECTOR STORE RETRIEVAL

 Testing retrieval with sample queries:

 Query: What are the most common complaints about credit cards?
  Top 3 results:
    1. Product: Credit card
       Issue: Fees or interest
       Similarity: 0.7959
       Preview: had no clue of the increase of interest of my capitol one credit cards...

    2. Product: Credit card
       Issue: Other features, terms, or problems
       Similarity: 0.7310
       Preview: chase bank credit card decreased all credit cards limit to all three credit cards. i have been payin...

    3. Product: Credit card
       Issue: Closing your account
       Similarity: 0.7216
       Preview: . it seems company s should at least be required to give more notice to the customer. thankfully i h...


 Query: Why are customers unhappy with money transfers?
  Top 3 results:
    1. Product: Credit card
       Issue: Problem when making payments
       Similarity: 0.5915
       Preview: . i should not have to wait a month for a refund 